<a href="https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("FlyRank-ML")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [3]:
con = duckdb.connect()

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*


### Distribution observations

The March 2026 data shows highly skewed distributions across the main search signals.

- `gsc_impressions` has a median of 0, a mean of 28.52, and a maximum of 40,084, indicating that a small number of content rows account for much higher search visibility.
- `gsc_clicks` is also heavily skewed, with a median of 0, a mean of 0.0835, and a maximum of 274.
- `gsc_avg_position` has a median of 7.5 compared with a mean of 15.83 and a maximum of 498, indicating a long right tail.

These distributions suggest that raw volume signals should be interpreted carefully when designing a baseline action rule.

In [4]:
# Section 1 — Distributions
# Inspect the distributions of the key signals used for the lane.

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

distribution_check = con.execute(f"""
    SELECT
        COUNT(*) AS n,

        MIN(gsc_impressions) AS impressions_min,
        AVG(gsc_impressions) AS impressions_mean,
        MEDIAN(gsc_impressions) AS impressions_median,
        MAX(gsc_impressions) AS impressions_max,

        MIN(gsc_avg_position) AS position_min,
        AVG(gsc_avg_position) AS position_mean,
        MEDIAN(gsc_avg_position) AS position_median,
        MAX(gsc_avg_position) AS position_max,

        MIN(gsc_clicks) AS clicks_min,
        AVG(gsc_clicks) AS clicks_mean,
        MEDIAN(gsc_clicks) AS clicks_median,
        MAX(gsc_clicks) AS clicks_max

    FROM read_parquet('{march_path}')
""").df()

distribution_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,impressions_min,impressions_mean,impressions_median,impressions_max,position_min,position_mean,position_median,position_max,clicks_min,clicks_mean,clicks_median,clicks_max
0,9841378,0,28.518119,0.0,40084,0.0,15.826651,7.5,498.0,0,0.083508,0.0,274


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal test #1 — Search volume

**Signal:** `gsc_impressions`

**Mini-test:** Compare click activity across impression-volume buckets.

**Verdict: CONFIRMED**

The data supports search volume as a useful directional signal. Average clicks increase consistently as impressions increase, from 0 clicks for rows with zero impressions to 5.07 average clicks for rows with more than 1,000 impressions. This indicates that higher-impression content has more observable search demand and may provide a larger opportunity for an action-oriented review.



In [5]:
# Signal test #1 — Search volume
# Check how clicks are distributed across impression-volume buckets.

signal_1 = con.execute(f"""
    SELECT
        CASE
            WHEN gsc_impressions = 0 THEN '0'
            WHEN gsc_impressions <= 10 THEN '1-10'
            WHEN gsc_impressions <= 100 THEN '11-100'
            WHEN gsc_impressions <= 1000 THEN '101-1000'
            ELSE '>1000'
        END AS impressions_bucket,

        COUNT(*) AS n,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_clicks) AS avg_clicks

    FROM read_parquet('{march_path}')

    GROUP BY 1

    ORDER BY
        CASE impressions_bucket
            WHEN '0' THEN 1
            WHEN '1-10' THEN 2
            WHEN '11-100' THEN 3
            WHEN '101-1000' THEN 4
            WHEN '>1000' THEN 5
        END
""").df()

signal_1

,impressions_bucket,n,total_impressions,total_clicks,avg_clicks
0,0,6230317,0.0,0.0,0.000000
1,1-10,1531634,6034759.0,16452.0,0.010741
2,11-100,1445944,54646247.0,157521.0,0.108940
3,101-1000,601123,161155930.0,483669.0,0.804609
4,>1000,32360,58820653.0,164190.0,5.073857


### Signal test #2 — Search position

**Signal:** `gsc_avg_position`

**Mini-test:** Compare click activity across average-position buckets.

**Verdict: CONFIRMED**

The data supports search position as a useful directional signal. Average clicks are highest for the >3–10 position bucket and then decline as position becomes worse, falling sharply from 0.121 for positions >20–50 to 0.005 for positions >50. This suggests that pages with worse search positions generally have lower observed search activity and that position can help prioritize review candidates.

In [6]:
# Signal test #2 — Search position
# Check how search activity varies across average-position buckets.

signal_2 = con.execute(f"""
    SELECT
        CASE
            WHEN gsc_avg_position IS NULL THEN 'NULL'
            WHEN gsc_avg_position <= 3 THEN '0-3'
            WHEN gsc_avg_position <= 10 THEN '>3-10'
            WHEN gsc_avg_position <= 20 THEN '>10-20'
            WHEN gsc_avg_position <= 50 THEN '>20-50'
            ELSE '>50'
        END AS position_bucket,

        COUNT(*) AS n,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_clicks) AS avg_clicks

    FROM read_parquet('{march_path}')

    GROUP BY 1

    ORDER BY
        CASE position_bucket
            WHEN '0-3' THEN 1
            WHEN '>3-10' THEN 2
            WHEN '>10-20' THEN 3
            WHEN '>20-50' THEN 4
            WHEN '>50' THEN 5
            WHEN 'NULL' THEN 6
        END
""").df()

signal_2

,position_bucket,n,total_impressions,total_clicks,avg_clicks
0,0-3,727362,54028594.0,205457.0,0.282469
1,>3-10,1456122,137830113.0,445828.0,0.306175
2,>10-20,519223,29386006.0,92449.0,0.178053
3,>20-50,631491,55944412.0,76589.0,0.121283
4,>50,276863,3468464.0,1509.0,0.005450
5,NULL,6230317,0.0,0.0,0.000000


### Signal test #3 — CTR vs search position

**Signal:** `gsc_ctr`

**Mini-test:** Compare CTR across search-position buckets to check whether CTR provides a meaningful signal for prioritization.

**Verdict: CONFIRMED**

CTR decreases consistently as search position becomes worse. CTR is highest for positions 0–3 (0.003803) and declines to 0.000435 for positions above 50. This supports CTR as a useful directional signal when evaluating search-performance opportunities.

In [7]:
# Signal test #3 — CTR vs search position

signal_3 = con.execute(f"""
    SELECT
        CASE
            WHEN gsc_avg_position IS NULL THEN 'NULL'
            WHEN gsc_avg_position <= 3 THEN '0-3'
            WHEN gsc_avg_position <= 10 THEN '>3-10'
            WHEN gsc_avg_position <= 20 THEN '>10-20'
            WHEN gsc_avg_position <= 50 THEN '>20-50'
            ELSE '>50'
        END AS position_bucket,

        COUNT(*) AS n,

        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions)
            ELSE 0
        END AS ctr

    FROM read_parquet('{march_path}')

    GROUP BY 1

    ORDER BY
        CASE position_bucket
            WHEN '0-3' THEN 1
            WHEN '>3-10' THEN 2
            WHEN '>10-20' THEN 3
            WHEN '>20-50' THEN 4
            WHEN '>50' THEN 5
            WHEN 'NULL' THEN 6
        END
""").df()

signal_3

,position_bucket,n,total_impressions,total_clicks,ctr
0,0-3,727362,54028594.0,205457.0,0.003803
1,>3-10,1456122,137830113.0,445828.0,0.003235
2,>10-20,519223,29386006.0,92449.0,0.003146
3,>20-50,631491,55944412.0,76589.0,0.001369
4,>50,276863,3468464.0,1509.0,0.000435
5,NULL,6230317,0.0,0.0,0.000000


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test — CTR-fix logic

**Flag-linked signal:** CTR versus search position

**Rule assumption:** Pages with relatively strong search positions but weak CTR may represent opportunities for CTR improvement.

**Verdict: CONFIRMED**

Among pages ranking within the top 10 positions, CTR varies substantially across the buckets. Pages with 0 CTR account for 1,869,166 rows, while the aggregate CTR increases from 0.0634% in the 0–0.1% bucket to 0.8893% for pages above 0.3% CTR.

This supports CTR as a useful signal for identifying potential CTR-improvement candidates among pages that already have relatively strong search positions. However, the test does not establish that every low-CTR page requires a CTR fix; it only confirms that CTR provides a meaningful signal for the rule.

In [8]:
# Flag-linked test — CTR vs position
# Focus on pages ranking within the top 10 positions.

flag_linked_test = con.execute(f"""
    SELECT
        CASE
            WHEN gsc_clicks = 0 THEN '0 CTR'
            WHEN gsc_clicks::DOUBLE / NULLIF(gsc_impressions, 0) <= 0.001
                THEN '0-0.1%'
            WHEN gsc_clicks::DOUBLE / NULLIF(gsc_impressions, 0) <= 0.003
                THEN '0.1-0.3%'
            ELSE '>0.3%'
        END AS ctr_bucket,

        COUNT(*) AS n,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions)
            ELSE 0
        END AS aggregate_ctr

    FROM read_parquet('{march_path}')

    WHERE gsc_avg_position <= 10

    GROUP BY 1

    ORDER BY
        CASE ctr_bucket
            WHEN '0 CTR' THEN 1
            WHEN '0-0.1%' THEN 2
            WHEN '0.1-0.3%' THEN 3
            WHEN '>0.3%' THEN 4
        END
""").df()

flag_linked_test

,ctr_bucket,n,total_impressions,total_clicks,aggregate_ctr
0,0 CTR,1869166,89727667.0,0.0,0.000000
1,0-0.1%,3703,7124014.0,4518.0,0.000634
2,0.1-0.3%,35672,28727696.0,57330.0,0.001996
3,>0.3%,274943,66279330.0,589437.0,0.008893


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*


Content teams should prioritize reviewing pages that already rank relatively well but receive very few clicks, since low CTR can indicate an opportunity to improve how the page attracts searchers. Impressions and search position can provide additional context, while CTR helps distinguish pages with strong visibility but weak click engagement.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.